# neuro-analog Quick Start Tour

This notebook provides a high-level overview of the neuro-analog framework for both hardware architects and neural architects.

**For hardware architects:** Learn what circuit primitives can implement neural computations and how to evaluate amenability of computational patterns.

**For neural architects:** Learn how your model behaves on analog hardware and how to measure degradation, energy savings, and latency.

**What this notebook covers:**
- Basic analog conversion and sweeps
- Intermediate representation concepts
- Amenability analysis
- Architecture families overview
- Circuit computation modes
- Ark export basics

**For deeper dives:** See the specialized notebooks linked throughout, and architecture-specific workflows in `examples/`.

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
_ROOT = Path.cwd().parent
sys.path.insert(0, str(_ROOT))

import torch
import torch.nn as nn
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

from neuro_analog.simulator import (
    analogize,
    mismatch_sweep,
    adc_sweep,
    ablation_sweep,
    count_analog_vs_digital,
)
from neuro_analog.ir.energy_model import HardwareProfile

## Why Analog Matters for Energy-Efficient AI

Digital neural networks are energy-hungry. Analog compute offers 10-100x energy reduction:

- **Digital MAC:** ~100 pJ per operation (GPU/SRAM-IMC baselines)
- **Analog MAC:** ~5 pJ per operation (IBM PCM arrays, crossbar physics)

This notebook shows how to evaluate whether your model can benefit from analog deployment.

In [ ]:
## 1. Create a Simple Model

We'll use a small MLP for demonstration.

class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)

model = TinyMLP()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
## 2. Train on Synthetic Data

# Generate synthetic data
X, y = make_classification(n_samples=2000, n_features=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

X_tr = torch.tensor(X_tr, dtype=torch.float32)
X_te = torch.tensor(X_te, dtype=torch.float32)
y_tr = torch.tensor(y_tr, dtype=torch.long)
y_te = torch.tensor(y_te, dtype=torch.long)

# Train
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(100):
    opt.zero_grad()
    loss = criterion(model(X_tr), y_tr)
    loss.backward()
    opt.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: loss = {loss.item():.4f}")

# Evaluate digital baseline
model.eval()
with torch.no_grad():
    pred = model(X_te).argmax(dim=1)
    digital_acc = (pred == y_te).float().mean().item()
print(f"\nDigital accuracy: {digital_acc*100:.1f}%")

In [ ]:
## 3. Convert to Analog

The `analogize()` function replaces PyTorch layers with analog equivalents.

# Convert to analog with 5% mismatch and 8-bit ADC
analog_model = analogize(model, sigma_mismatch=0.05, n_adc_bits=8)

# Check what was converted
counts = count_analog_vs_digital(analog_model)
print("Analog model composition:")
print(f"  Analog layers:   {counts['analog_layers']}")
print(f"  Analog params:   {counts['analog_params']:,}")
print(f"  Coverage:        {counts['coverage_pct']:.1f}%")
print(f"  Digital layers:  {counts['digital_layers']}")
print(f"  Digital params:  {counts['digital_params']:,}")

In [ ]:
## 4. Run Mismatch Sweep

Sweep over different noise levels to see how accuracy degrades.

def eval_fn(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te).argmax(dim=1)
    return float((pred == y_te).float().mean())

# Create hardware profile for energy/latency estimation
profile = HardwareProfile()

# Run sweep
sweep = mismatch_sweep(
    model,
    eval_fn,
    sigma_values=[0.0, 0.02, 0.05, 0.07, 0.10, 0.12, 0.15],
    n_trials=30,
    n_adc_bits=8,
    hardware_profile=profile,
)

print("\nMismatch sweep results:")
print(f"{'Sigma':>8} {'Accuracy':>10} {'Std':>8} {'Retained':>10}")
for i, sigma in enumerate(sweep.sigma_values):
    print(f"{sigma:8.2f} {sweep.mean[i]*100:9.1f}% {sweep.std[i]*100:7.1f}% {sweep.normalized_mean[i]:9.1%}")

In [ ]:
## 5. Energy and Latency Estimates

print("\nEnergy and latency estimates:")
print(f"  Analog energy:   {sweep.analog_energy_pJ:,.0f} pJ")
print(f"  Digital energy:  {sweep.digital_energy_pJ:,.0f} pJ")
print(f"  Energy saving:    {sweep.energy_saving_vs_digital*100:.1f}%")
print(f"\n  Analog latency:  {sweep.analog_latency_ns:,.0f} ns")
print(f"  Digital latency: {sweep.digital_latency_ns:,.0f} ns")
print(f"  Speedup:          {sweep.speedup_vs_digital:.1f}x")

In [ ]:
## 6. Degradation Threshold

The degradation threshold is the noise level where accuracy drops by 10%.

threshold = sweep.degradation_threshold(max_relative_loss=0.10)
print(f"\n10% degradation threshold: sigma = {threshold:.3f}")
print(f"This means the model tolerates up to {threshold*100:.1f}% conductance mismatch.")

In [ ]:
## 7. ADC Precision Sweep

See how ADC bit width affects accuracy at fixed mismatch.

adc = adc_sweep(
    model,
    eval_fn,
    bit_values=[2, 4, 6, 8, 10, 12],
    n_trials=20,
    sigma_mismatch=0.05,
    hardware_profile=profile,
)

print("\nADC precision sweep results:")
print(f"{'Bits':>6} {'Accuracy':>10} {'Std':>8} {'Retained':>10}")
for i, bits in enumerate(adc.sigma_values):
    print(f"{int(bits):6d} {adc.mean[i]*100:9.1f}% {adc.std[i]*100:7.1f}% {adc.normalized_mean[i]:9.1%}")

In [ ]:
## 8. Noise Ablation

Isolate which noise source (mismatch, thermal, quantization) matters most.

ablation = ablation_sweep(
    model,
    eval_fn,
    sigma_values=[0.0, 0.05, 0.10],
    n_trials=30,
    hardware_profile=profile,
)

print("\nNoise attribution at sigma=0.10:")
for noise_type, result in ablation.items():
    retained = result.normalized_mean[-1]
    print(f"  {noise_type:12s}: {retained:.1%} retained")

## Key Takeaways from Sweeps

1. **Analog conversion is automatic** - `analogize()` handles layer replacement
2. **Energy savings are significant** - analog MACs are ~20x more efficient than digital (5 pJ vs 100 pJ)
3. **Mismatch is the dominant noise source** - thermal noise and quantization have smaller effects
4. **Degradation threshold tells you the tolerance** - higher is better for hardware design

## 4. Intermediate Representation (IR)

The neuro-analog framework uses an Intermediate Representation to decompose models into circuit-level primitives. This enables accurate energy estimation and hardware annotation.

In [ ]:
from neuro_analog.ir import AnalogGraph, make_mvm_node, make_activation_node
from neuro_analog.ir.types import OpType, Domain

# Build a minimal IR graph manually
graph = AnalogGraph(name="minimal_example")

# Add a matrix-vector multiply (crossbar operation)
mvm = make_mvm_node(
    name="fc1",
    input_shape=(20,),
    output_shape=(64,),
    flops=1280,  # 20 * 64
)
graph.add_node(mvm)

# Add an activation (Tanh maps to subthreshold MOSFET differential pair)
act = make_activation_node(
    name="tanh1",
    input_shape=(64,),
    output_shape=(64,),
    op_type=OpType.ANALOG_SIGMOID,  # Analog-native activation
    domain=Domain.ANALOG,
)
graph.add_node(act)

print(f"Graph has {len(graph.nodes)} nodes")
print(f"Node 0: {graph.nodes[0].op_type.name} in {graph.nodes[0].domain.name} domain")
print(f"Node 1: {graph.nodes[1].op_type.name} in {graph.nodes[1].domain.name} domain")

**Key IR concepts:**

- **OpType:** ~20 circuit-level primitives (MVM, INTEGRATION, DECAY, SOFTMAX, etc.)
- **Domain:** ANALOG (runs in analog), DIGITAL (requires digital), HYBRID (analog-possible with quality loss)
- **D/A boundaries:** Each ADC/DAC crossing costs energy and accumulates quantization error

**For a deep dive:** See `intermediate_representation.ipynb`

## 5. Amenability Analysis

Amenability scoring combines multiple factors to predict how well an architecture will work on analog hardware.

In [ ]:
from neuro_analog.ir.types import AnalogAmenabilityProfile, ArchitectureFamily, DynamicsProfile
from neuro_analog.analysis.compute_amenability import compute_amenability_score

# Create a simple amenability profile
profile = AnalogAmenabilityProfile(
    architecture=ArchitectureFamily.TRANSFORMER,
    model_name="tiny_mlp",
    model_params=sum(p.numel() for p in model.parameters()),
    analog_flop_fraction=0.8,  # 80% of compute in analog
    digital_flop_fraction=0.2,
    hybrid_flop_fraction=0.0,
    da_boundary_count=2,  # 2 ADC/DAC crossings
    layer_count=3,
    sigma_10pct=0.10,  # 10% mismatch threshold from sweep
    dynamics=DynamicsProfile(has_dynamics=False),
)

# Compute amenability score
score = compute_amenability_score(profile)
print(f"Amenability score: {score:.2f} (0-1, higher = more analog-amenable)")

# Classify failure mode
from neuro_analog.analysis.design_heuristics import classify_failure_mode
failure_mode = classify_failure_mode(profile)
print(f"Failure mode: {failure_mode}")

**Amenability combines:**
- Analog FLOP fraction (more analog = better)
- Noise tolerance (sigma_10pct from sweeps)
- D/A boundary count (fewer = better)
- Dynamics penalty (iterative/multi-step = worse)
- Precision requirements (lower bits = better)

**Failure modes:**
- **Single-pass tolerant:** High analog fraction, low boundary density, good noise tolerance
- **Fixed-point sensitive (DEQ):** Iterative convergence compounds mismatch
- **Diffusion-like:** Multi-step pipeline accumulates quantization error

**For a deep dive:** See `amenability_analysis.ipynb`

## 6. Architecture Families

Different neural network architectures have different analog amenability based on their computational structure.

In [ ]:
# The 7 architecture families and their amenability ranking
families = [
    ("Neural ODE", "Highest - ODE form maps directly to RC integrators"),
    ("SSM (Mamba)", "High - Linear recurrence, minimal D/A boundaries"),
    ("EBM", "High - Energy landscape wide, noise just blurs attractors"),
    ("Flow (FLUX)", "Medium - Many function evaluations accumulate error"),
    ("Transformer", "Medium - FFN analog partition, softmax digital-required"),
    ("DEQ", "Low - Fixed-point iteration compounds mismatch"),
    ("Diffusion", "Lowest - 100+ steps accumulate quantization error"),
]

print("Architecture amenability ranking:")
for i, (name, reason) in enumerate(families, 1):
    print(f"{i}. {name:20s} - {reason}")

**Why ranking matters:** Single-pass architectures (noise doesn't accumulate) tolerate analog nonidealities better than iterative (DEQ) or multi-step (Diffusion) architectures.

**For a deep dive:** See `architecture_families.ipynb`  
**For cross-arch comparison:** See `examples/02_seven_arch_sweep.py`

## 7. Circuit Computation Modes

The framework supports different analog circuit modes for different architecture families.

In [ ]:
# Circuit modes and their physical implementations
circuit_modes = [
    ("rc_integrator", "Neural ODE, Flow", "RC circuit ODE solver with Johnson-Nyquist noise"),
    ("hopfield", "DEQ", "Continuous-time analog feedback relaxation"),
    ("classic", "Diffusion", "DDIM (deterministic) diffusion"),
    ("cld", "Diffusion", "Critically-damped Langevin dynamics"),
]

print("Circuit computation modes:")
for mode, arch, description in circuit_modes:
    print(f"  {mode:15s} ({arch:15s}): {description}")

print("\nNote: These modes measure true analog hyperefficiency, not digital approximations.")

## 8. Ark Export (Optional/Advanced)

Ark is an analog circuit compiler that can export ODE-based models to real analog circuits.

In [ ]:
# Ark export example (requires Ark installation)
# from neuro_analog.ark_bridge.neural_ode_cdg import export_neural_ode_to_ark
# 
# # Export a Neural ODE model to Ark
# ark_circuit = export_neural_ode_to_ark(neural_ode_model)
# print(f"Generated BaseAnalogCkt subclass: {ark_circuit.__name__}")
# 
# # Run nominal and mismatch forward passes in Ark
# nominal_out = ark_circuit.forward(x)
# mismatch_out = ark_circuit.forward_with_mismatch(x, sigma=0.05)

print("Ark export requires Ark installation (JAX, diffrax).")
print("See ark_export.ipynb for a detailed walkthrough.")

**For a deep dive:** See `ark_export.ipynb`  
**For full pipeline:** See `examples/03_ark_pipeline.py`

## 9. Next Steps

This notebook provided a high-level overview. Here's how to continue based on your goals:

**For hardware architects:**
- `intermediate_representation.ipynb` - Deep dive into circuit primitives, graph construction, and hardware annotation
- `amenability_analysis.ipynb` - Detailed scoring, failure modes, and design recommendations for computational patterns
- `architecture_families.ipynb` - Computational patterns and circuit mode mappings your hardware should support

**For neural architects:**
- `amenability_analysis.ipynb` - Understand how your model will behave on analog
- `ark_export.ipynb` - Export your trained model to analog circuits (if ODE-based)
- `examples/` directory - Architecture-specific workflows for your model family

**For both:**
- `examples/02_seven_arch_sweep.py` - Cross-architecture comparison and ranking
- `examples/03_ark_pipeline.py` - Full Ark pipeline for Neural ODEs
- `examples/04_amenability_scoring.py` - Energy/latency modeling example